# Day 8 v1 — Silver Transformation: Payments Only
**Single entity, step-by-step — learning version**

**Source:** `/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/api/payments/`
**Sink:** `/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api/payments/` (Delta)

This notebook does ONE entity (payments) with no helper functions — every step is written out explicitly so you can see exactly what is happening.

**Steps:**
1. Read Bronze JSON files for payments
2. Explode the `data[]` array from the API wrapper
3. Cast columns to correct types
4. Add Silver audit columns
5. Deduplicate on `payment_id`
6. Write to Silver as Delta table
7. Verify the output

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Constants
BRONZE_PAYMENTS = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/api/payments"
SILVER_PAYMENTS = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api/payments"

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze path : {BRONZE_PAYMENTS}")
print(f"Silver path : {SILVER_PAYMENTS}")
print(f"Run time    : {RUN_TS}")

In [ ]:
# Cell 3 — Read all Bronze JSON files for payments
# Bronze stores one JSON file per API page, per ingestion date:
#   payments/ingestion_date=2026-07-13/page_1.json
#   payments/ingestion_date=2026-07-13/page_2.json
#   payments/ingestion_date=2026-07-14/page_1.json  ...
#
# spark.read.json() reads ALL files under the path recursively.
# multiline=true because each JSON file is a single object, not one object per line.

raw_df = (
    spark.read
    .option("multiline", "true")
    .json(BRONZE_PAYMENTS)
)

print(f"Files read. Schema:")
raw_df.printSchema()
print(f"Row count (one row = one JSON page file): {raw_df.count()}")

In [ ]:
# Cell 4 — Explode the data[] array
# Each JSON file has this structure:
#   { "data": [ {record1}, {record2}, ... ], "pagination": { ... } }
#
# explode() turns the array into individual rows — one row per payment record.
# select("record.*") flattens the struct so every field becomes its own column.

exploded_df = raw_df.select(F.explode(F.col("data")).alias("record"))
flat_df     = exploded_df.select("record.*")

print(f"After explode — row count (one row = one payment): {flat_df.count()}")
print()
flat_df.printSchema()
flat_df.show(3, truncate=True)

In [ ]:
# Cell 5 — Cast columns to correct types
# Bronze JSON stores everything as strings.
# We explicitly cast each column to its correct data type.

typed_df = flat_df.select(
    F.col("payment_id").cast("string"),
    F.col("session_id").cast("string"),
    F.col("customer_id").cast("string"),
    F.col("amount_aud").cast("decimal(10,2)"),      # money — 2 decimal places
    F.col("status").cast("string"),
    F.col("payment_mode").cast("string"),
    F.col("created_at").cast("timestamp"),      # ISO string -> timestamp
    F.col("updated_at").cast("timestamp"),
)

print("After type casting — schema:")
typed_df.printSchema()
typed_df.display()

In [ ]:
# Cell 6 — Add Silver audit columns
# These columns are added to EVERY Silver table — same pattern across all entities.
# They tell us: when was this record loaded into Silver, and by which pipeline.

audited_df = (
    typed_df
    .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
    .withColumn("silver_load_type",   F.lit("full"))
    .withColumn("silver_pipeline",    F.lit("pl_silver_payments_v1"))
)

print("After adding audit columns:")
audited_df.printSchema()

In [ ]:
# Cell 7 — Deduplicate on payment_id
# Bronze can contain the same payment_id across multiple ingestion dates
# (incremental runs overlap slightly, or a record was updated).
#
# Window function: for each payment_id, rank rows by updated_at descending.
# Row 1 = most recent version of that payment. We keep only row 1.

window = Window.partitionBy("payment_id").orderBy(F.col("updated_at").desc())

deduped_df = (
    audited_df
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

before = audited_df.count()
after  = deduped_df.count()
print(f"Before dedup : {before} rows")
print(f"After dedup  : {after} rows")
print(f"Duplicates removed : {before - after}")

In [ ]:
# Cell 8 — Write to Silver as Delta table (full overwrite)
# Delta format gives us: ACID transactions, schema enforcement,
# time travel, and efficient upserts (used in v3 incremental).
#
# mode("overwrite") replaces the entire Silver table — safe for first load.

(
    deduped_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PAYMENTS)
)

print(f"Written to Silver: {SILVER_PAYMENTS}")
print(f"Rows written: {deduped_df.count()}")

In [ ]:
# Cell 9 — Verify Silver output
# Read back from Silver and confirm the data looks correct.

silver_df = spark.read.format("delta").load(SILVER_PAYMENTS)

print("Silver payments — schema:")
silver_df.printSchema()

print(f"\nTotal rows in Silver: {silver_df.count()}")
print("\nSample rows:")
silver_df.show(5, truncate=True)

print("\nNull check on payment_id (should be 0):")
silver_df.filter(F.col("payment_id").isNull()).count()

print("\nStatus distribution:")
silver_df.groupBy("status").count().orderBy("count", ascending=False).show()